# Agent-Based Modeling for Innovation Diffusion: A Comprehensive Case Study

This notebook demonstrates how to use the `innovate` library's agent-based modeling (ABM) capabilities to simulate innovation adoption in complex social networks. We'll explore competitive dynamics, network effects, and policy interventions through practical examples.

## Table of Contents

1. **Introduction to ABM in Innovation Diffusion**
2. **Competitive Innovation Diffusion**
3. **Disruptive Innovation Dynamics**
4. **Network Structure Effects**
5. **Policy Intervention Simulations**
6. **Real-world Case Study: Social Media Platform Adoption**

---

## 1. Introduction to ABM in Innovation Diffusion

Agent-based models allow us to simulate innovation adoption from the bottom-up, where individual agents make decisions based on local interactions, network effects, and personal characteristics. This approach captures emergent phenomena that aggregate models might miss.

### Key Advantages of ABM:

- **Heterogeneous Agents**: Different adoption thresholds, influence levels, and preferences
- **Network Effects**: Spatial and social network structures affect diffusion patterns
- **Non-linear Dynamics**: Tipping points, path dependence, and complex feedback loops
- **Policy Testing**: Simulate interventions before real-world implementation


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector
import networkx as nx

# Import innovate ABM components
from innovate.abm import (
    InnovationAgent, 
    CompetitiveDiffusionAgent, 
    CompetitiveDiffusionModel,
    DisruptiveInnovationAgent,
    DisruptiveInnovationModel
)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Competitive Innovation Diffusion

Let's start with a scenario where multiple innovations compete for adoption in a spatial network. This could represent competing technologies, standards, or products.

In [ ]:
# Create and run a competitive diffusion model
def run_competitive_diffusion_simulation():
    # Model parameters
    num_agents = 400
    grid_width = 20
    grid_height = 20
    num_innovations = 3
    n_steps = 50
    
    # Create the model
    model = CompetitiveDiffusionModel(
        num_agents=num_agents,
        width=grid_width,
        height=grid_height,
        num_innovations=num_innovations
    )
    
    # Run the simulation
    results_df = model.run_model(n_steps)
    
    return model, results_df

# Run the simulation
competitive_model, competitive_results = run_competitive_diffusion_simulation()
print("Competitive diffusion simulation completed!")
print(f"Final step adoption counts: {competitive_results['AdoptionCounts'].iloc[-1]}")

In [ ]:
# Visualize competitive diffusion results
def plot_competitive_adoption(results_df):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Extract adoption counts for each innovation
    adoption_data = []
    for step, counts in enumerate(results_df['AdoptionCounts']):
        for innovation, count in enumerate(counts):
            adoption_data.append({
                'Step': step,
                'Innovation': f'Innovation {innovation}',
                'Adopters': count
            })
    
    adoption_df = pd.DataFrame(adoption_data)
    
    # Time series plot
    for innovation in adoption_df['Innovation'].unique():
        data = adoption_df[adoption_df['Innovation'] == innovation]
        ax1.plot(data['Step'], data['Adopters'], marker='o', label=innovation, linewidth=2)
    
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Number of Adopters')
    ax1.set_title('Competitive Innovation Adoption Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Market share pie chart (final step)
    final_counts = results_df['AdoptionCounts'].iloc[-1]
    labels = [f'Innovation {i}' for i in range(len(final_counts))]
    ax2.pie(final_counts, labels=labels, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Final Market Share Distribution')
    
    plt.tight_layout()
    plt.show()

plot_competitive_adoption(competitive_results)

### Analyzing Network Effects

The spatial arrangement of agents creates network effects where nearby agents influence each other's adoption decisions. Let's examine how different network structures affect diffusion patterns.

In [ ]:
# Function to visualize agent positions and adoptions
def visualize_agent_grid(model, title="Agent Adoption Status"):
    """
    Visualize the spatial distribution of agents and their adoption status
    """
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Create a grid for visualization
    grid_vis = np.full((model.grid.height, model.grid.width), -2, dtype=int)
    
    # Fill in agent positions and adoptions
    for agent in model.agents:
        x, y = agent.pos
        if hasattr(agent, 'adopted_innovation'):
            grid_vis[y, x] = agent.adopted_innovation
        elif hasattr(agent, 'choice'):
            grid_vis[y, x] = 1 if agent.choice == 'disruptive' else 0
    
    # Create colormap
    colors = ['lightgray', 'red', 'blue', 'green', 'orange', 'purple']
    bounds = [-2.5, -0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
    from matplotlib.colors import ListedColormap, BoundaryNorm
    cmap = ListedColormap(colors[:len(set(grid_vis.flatten()))])
    norm = BoundaryNorm(bounds[:len(set(grid_vis.flatten()))+1], cmap.N)
    
    im = ax.imshow(grid_vis, cmap=cmap, norm=norm, origin='lower')
    ax.set_title(title)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    
    # Add colorbar
    unique_values = sorted(set(grid_vis.flatten()))
    labels = ['No Agent' if x == -2 else f'Innovation {x}' if x >= 0 else 'No Adoption' 
             for x in unique_values]
    cbar = plt.colorbar(im, ax=ax, ticks=unique_values)
    cbar.ax.set_yticklabels(labels)
    
    plt.tight_layout()
    plt.show()

# Visualize the final state of competitive diffusion
visualize_agent_grid(competitive_model, "Final Competitive Diffusion State")

## 3. Disruptive Innovation Dynamics

Now let's examine how disruptive innovations can gradually overtake incumbent technologies through performance improvements over time.

In [ ]:
# Disruptive innovation simulation
def run_disruptive_innovation_simulation():
    model = DisruptiveInnovationModel(
        num_agents=300,
        width=20,
        height=15,
        initial_disruptive_performance=0.3,
        disruptive_performance_improvement=0.02
    )
    
    results = model.run_model(50)
    return model, results

disruptive_model, disruptive_results = run_disruptive_innovation_simulation()
print('Disruptive innovation simulation completed!')

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot(disruptive_results.index, disruptive_results['IncumbentAdopters'], 
         label='Incumbent', marker='o', linewidth=2)
ax1.plot(disruptive_results.index, disruptive_results['DisruptiveAdopters'], 
         label='Disruptive', marker='s', linewidth=2)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Number of Adopters')
ax1.set_title('Disruptive Innovation Adoption Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Performance evolution
steps = range(len(disruptive_results))
incumbent_perf = [1.0] * len(steps)
disruptive_perf = [0.3 + 0.02 * i for i in steps]

ax2.plot(steps, incumbent_perf, label='Incumbent Performance', linewidth=2)
ax2.plot(steps, disruptive_perf, label='Disruptive Performance', linewidth=2)
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Price Threshold')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Performance Level')
ax2.set_title('Technology Performance Evolution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Policy Intervention Analysis

Let's create a custom ABM to test how policy interventions (subsidies, regulations, information campaigns) affect innovation diffusion.

In [ ]:
class PolicyAgent(InnovationAgent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.adoption_threshold = np.random.uniform(0.1, 0.8)
        self.policy_sensitivity = np.random.uniform(0.5, 1.5)
    
    def step(self):
        if self.adopted:
            return
            
        # Calculate adoption probability based on neighbors and policy
        neighbors = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        if neighbors:
            adoption_rate = sum(1 for n in neighbors if n.adopted) / len(neighbors)
        else:
            adoption_rate = 0
            
        # Policy effect (subsidy reduces effective threshold)
        effective_threshold = self.adoption_threshold - (self.model.policy_strength * self.policy_sensitivity)
        
        if adoption_rate > effective_threshold:
            self.adopted = True

class PolicyInterventionModel(Model):
    def __init__(self, num_agents, width, height, policy_strength=0.0):
        super().__init__()
        self.num_agents = num_agents
        self.policy_strength = policy_strength  # 0 = no policy, 1 = strong policy
        self.grid = MultiGrid(width, height, True)
        self.running = True
        
        # Create agents
        for i in range(num_agents):
            agent = PolicyAgent(i, self)
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(agent, (x, y))
            
        # Seed initial adopters (2% of population)
        initial_adopters = self.random.sample(list(self.agents), k=max(1, num_agents // 50))
        for agent in initial_adopters:
            agent.adopted = True
            
        self.datacollector = DataCollector(
            model_reporters={
                "Adopters": lambda m: sum(1 for a in m.agents if a.adopted),
                "AdoptionRate": lambda m: sum(1 for a in m.agents if a.adopted) / m.num_agents
            }
        )
        
    def step(self):
        self.datacollector.collect(self)
        self.agents.do("step")
        
# Run policy comparison
def compare_policy_interventions():
    policies = {'No Policy': 0.0, 'Weak Policy': 0.1, 'Strong Policy': 0.3}
    results = {}
    
    for policy_name, strength in policies.items():
        model = PolicyInterventionModel(400, 20, 20, policy_strength=strength)
        for _ in range(100):
            model.step()
        results[policy_name] = model.datacollector.get_model_vars_dataframe()
    
    return results

policy_results = compare_policy_interventions()

# Plot policy comparison
plt.figure(figsize=(12, 8))
for policy_name, data in policy_results.items():
    plt.plot(data.index, data['AdoptionRate'], label=policy_name, linewidth=2, marker='o')

plt.xlabel('Time Step')
plt.ylabel('Adoption Rate')
plt.title('Impact of Policy Interventions on Innovation Diffusion')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Policy Intervention Results:")
for policy_name, data in policy_results.items():
    final_adoption = data['AdoptionRate'].iloc[-1]
    print(f"{policy_name}: {final_adoption:.1%} final adoption rate")

## 5. Real-world Case Study: Social Media Platform Adoption

Let's model the adoption of competing social media platforms, incorporating network effects, switching costs, and platform improvements.

In [ ]:
class SocialMediaAgent(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.platform = None  # 'A', 'B', or None
        self.adoption_threshold = np.random.uniform(0.2, 0.7)
        self.switching_cost = np.random.uniform(0.1, 0.4)
        
    def step(self):
        neighbors = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        if not neighbors:
            return
            
        # Count platform adoption among neighbors
        platform_counts = {'A': 0, 'B': 0, None: 0}
        for neighbor in neighbors:
            platform_counts[neighbor.platform] += 1
            
        total_neighbors = len(neighbors)
        platform_a_rate = platform_counts['A'] / total_neighbors
        platform_b_rate = platform_counts['B'] / total_neighbors
        
        # Calculate utility for each platform (network effects + platform quality)
        utility_a = platform_a_rate + self.model.platform_a_quality
        utility_b = platform_b_rate + self.model.platform_b_quality
        
        # Decision logic
        if self.platform is None:
            # First-time adoption
            if utility_a > self.adoption_threshold and utility_a > utility_b:
                self.platform = 'A'
            elif utility_b > self.adoption_threshold and utility_b > utility_a:
                self.platform = 'B'
        else:
            # Potential switching
            current_utility = utility_a if self.platform == 'A' else utility_b
            alternative_utility = utility_b if self.platform == 'A' else utility_a
            
            if alternative_utility > current_utility + self.switching_cost:
                self.platform = 'B' if self.platform == 'A' else 'A'

class SocialMediaModel(Model):
    def __init__(self, num_agents, width, height):
        super().__init__()
        self.num_agents = num_agents
        self.grid = MultiGrid(width, height, True)
        self.running = True
        
        # Platform qualities (can evolve over time)
        self.platform_a_quality = 0.3
        self.platform_b_quality = 0.2
        
        # Create agents
        for i in range(num_agents):
            agent = SocialMediaAgent(i, self)
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(agent, (x, y))
            
        self.datacollector = DataCollector(
            model_reporters={
                "Platform_A": lambda m: sum(1 for a in m.agents if a.platform == 'A'),
                "Platform_B": lambda m: sum(1 for a in m.agents if a.platform == 'B'),
                "No_Platform": lambda m: sum(1 for a in m.agents if a.platform is None)
            }
        )
        
    def step(self):
        # Platform B improves quality over time (disruptive innovation)
        self.platform_b_quality += 0.005
        
        self.datacollector.collect(self)
        self.agents.do("step")

# Run social media simulation
social_model = SocialMediaModel(500, 25, 20)
for _ in range(150):
    social_model.step()

social_results = social_model.datacollector.get_model_vars_dataframe()

# Plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Adoption over time
ax1.plot(social_results.index, social_results['Platform_A'], label='Platform A (Incumbent)', linewidth=2)
ax1.plot(social_results.index, social_results['Platform_B'], label='Platform B (Challenger)', linewidth=2)
ax1.plot(social_results.index, social_results['No_Platform'], label='Non-adopters', linewidth=2)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Number of Users')
ax1.set_title('Social Media Platform Competition')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Platform quality evolution
steps = range(len(social_results))
quality_a = [0.3] * len(steps)
quality_b = [0.2 + 0.005 * i for i in steps]

ax2.plot(steps, quality_a, label='Platform A Quality', linewidth=2)
ax2.plot(steps, quality_b, label='Platform B Quality', linewidth=2)
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Platform Quality')
ax2.set_title('Platform Quality Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Key Takeaways and Best Practices

### What We've Learned:

1. **Network Effects Matter**: Spatial and social proximity significantly influence adoption patterns
2. **Competition Dynamics**: Multiple innovations compete based on relative attractiveness and network effects
3. **Disruptive Potential**: Lower-performing technologies can overtake incumbents through continuous improvement
4. **Policy Impact**: Interventions can accelerate or redirect diffusion patterns
5. **Switching Costs**: Existing adoption creates inertia that new technologies must overcome

### Best Practices for ABM in Innovation Research:

- **Validate Against Real Data**: Compare simulation results with historical adoption patterns
- **Sensitivity Analysis**: Test how results change with different parameter values
- **Multiple Runs**: Average results across multiple simulation runs for robust findings
- **Calibration**: Use empirical data to set realistic parameter ranges
- **Visualization**: Use spatial and temporal visualizations to understand emergence

### Next Steps:

- Integrate ABM results with aggregate diffusion models
- Add more realistic network structures (small-world, scale-free)
- Include economic factors (pricing, income distribution)
- Model organizational adoption and B2B diffusion
- Incorporate machine learning for agent decision-making
